# Exercise 4

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
import math

## Part 1

### Blocking simulation

In [4]:

def blocking(mean_arrival_time,mean_servive_time,customers,units):
    #These use mean times
    t_interArrival = np.random.exponential(mean_arrival_time,customers)
    t_arrival = np.cumsum(t_interArrival)
    t_service = np.random.exponential(mean_servive_time,customers)

    service_units = [0]*units
    n_blocked = 0
    for i in range(0,customers):
        arrival_time = t_arrival[i]
        t_departure = arrival_time+t_service[i]
        
        vacancies = [x for x in service_units if x < arrival_time]
        
        if len(vacancies) > 0:
            indx = service_units.index(vacancies[0])
            service_units[indx] = t_departure
            
        else:
            n_blocked += 1
      
    return n_blocked/customers




### Testing simulation

In [5]:
#Parameters
m = 10
mean_service_time = 8
mean_interarrival_time = 1
t_service_rate = 1/mean_service_time
interArrival_rate = 1/mean_interarrival_time

n_repetitions = 10
n_customers = 10000

print("B_simulation:",blocking(mean_interarrival_time,mean_service_time,n_customers,m))

B_simulation: 0.1135


### Comparing with theory

In [6]:
A = mean_interarrival_time * mean_service_time

num = (A**m)/math.factorial(m)

denom = 0
for i in range(0, m + 1):
    denom += (A**i) / math.factorial(i)

B = num / denom

print("B_theory:",B)

B_theory: 0.12166106425295149


### Calculating confidence interval

In [12]:
reps_list = []
for i in range(n_repetitions):
    reps_list.append(blocking(mean_interarrival_time, mean_service_time, n_customers, m))
reps = np.array(reps_list)



#Short method (same result):
conf_int = stats.t.interval(confidence = 0.95, df=n_repetitions-1, loc=np.mean(reps), scale=np.std(reps, ddof=1)/np.sqrt(n_repetitions))

#Long method (same result):
mean = np.mean(reps)

#var = np.var(reps, ddof=1)

std = np.std(reps, ddof=1)

alpha=0.05
p_quantile_1 = stats.t.ppf(q=alpha/2,df=n_repetitions-1)
p_quantile_2 = stats.t.ppf(q=1-alpha/2,df=n_repetitions-1)

conf_int = [mean + (std/np.sqrt(n_repetitions))*p_quantile_1,mean + (std/np.sqrt(n_repetitions))*p_quantile_2]
print(conf_int)

[np.float64(0.11599713243563459), np.float64(0.12656286756436538)]


### Part 2

#### General blocking function

In [13]:
def blocking_with_interarrival(t_interArrival, mean_service_time, customers, units):
    t_arrival = np.cumsum(t_interArrival)
    t_service = np.random.exponential(mean_service_time, customers)

    service_units = [0] * units
    n_blocked = 0

    for i in range(customers):
        arrival_time = t_arrival[i]
        departure_time = arrival_time + t_service[i]

        vacancies = [x for x in service_units if x < arrival_time]

        if len(vacancies) > 0:
            indx = service_units.index(vacancies[0])
            service_units[indx] = departure_time
        else:
            n_blocked += 1

    return n_blocked / customers

#### Erlang distribution of arrivals

In [14]:
def erlang_interarrival(customers, k=10, mean = 1):
    # mean = k * theta <=> theta = mean/k 
    theta = mean/k
    return np.random.gamma(shape=k, scale=theta, size=customers)




#### hyperexponential inter-arrival distribution.

#### interarrival function

In [25]:
def hyperexponential_interarrival(customers):
    #set parameters
    p1 = 0.8
    #p2 = 0.2 #not needed just here to remember 
    lambda1 = 0.8333
    lambda2 = 5.0

    # Decides which exponential distribution each customer uses
    choices = np.random.uniform(0,1,customers)

    t_interArrival = np.empty(customers)

    use_first = choices < p1
    use_second = ~use_first #flips all values of use_first

    #mean arrival time = 1/lambda    
    mean_1 = 1/lambda1 
    mean_2 = 1/lambda2
    
    for i in range(customers):
        if use_first[i] == True:
            t_interArrival[i] = np.random.exponential(scale=mean_1)
        else:
            t_interArrival[i] = np.random.exponential(scale=mean_2)

    return t_interArrival



#### running blocking functions

In [26]:
t_interArrival_erlang = erlang_interarrival(n_customers)
print("Blocking erlang:",blocking_with_interarrival(t_interArrival_erlang, mean_service_time ,n_customers,m))


t_interArrival_hyper = hyperexponential_interarrival(n_customers)
print("Blocking with hyper", blocking_with_interarrival(t_interArrival_hyper, mean_service_time, n_customers, m))

Blocking erlang: 0.0711
Blocking with hyper 0.1302


#### Confidence intervals

In [27]:
def confidence_interval(inter_arrival_function):
    reps_list = []

    for i in range(n_repetitions):
        inter_arrival = inter_arrival_function(n_customers)
        reps_list.append(blocking_with_interarrival(inter_arrival, mean_service_time, n_customers, m))

    reps = np.array(reps_list)

    mean = np.mean(reps)
    conf_int = stats.t.interval(
        confidence=0.95,
        df=n_repetitions-1,
        loc=mean,
        scale=np.std(reps, ddof=1) / np.sqrt(n_repetitions)
    )

    return mean, conf_int

In [28]:
mean_hyper, ci_hyper = confidence_interval(hyperexponential_interarrival)
mean_erlang, ci_erlang = confidence_interval(erlang_interarrival)

print("Hyperexponential")
print("Mean blocking:", mean_hyper)
print("Confidence interval:", ci_hyper)

print("\nErlang")
print("Mean blocking:", mean_erlang)
print("Confidence interval:", ci_erlang)

Hyperexponential
Mean blocking: 0.13935
Confidence interval: (np.float64(0.13300481605552544), np.float64(0.14569518394447456))

Erlang
Mean blocking: 0.06599999999999999
Confidence interval: (np.float64(0.06465026728924392), np.float64(0.06734973271075606))


## Part 3

### Service time generator functions 

In [30]:
def constant_service(customers, mean_service_time=8):
    return np.full(customers, mean_service_time)


def pareto_service(customers, k, mean_service_time=8):
    beta = mean_service_time * (k - 1) / k
    U = np.random.uniform(0, 1, customers)
    return beta * U ** (-1 / k)


def erlang_service(customers, k=10, mean_service_time=8):
    theta = mean_service_time / k
    return np.random.gamma(shape=k, scale=theta, size=customers)


def hyperexponential_service(customers):
    p1 = 0.8
    lambda1 = 0.8333 / 8
    lambda2 = 5.0 / 8

    choices = np.random.uniform(0, 1, customers)
    service_times = np.empty(customers)

    for i in range(customers):
        if choices[i] < p1:
            service_times[i] = np.random.exponential(scale=1 / lambda1)
        else:
            service_times[i] = np.random.exponential(scale=1 / lambda2)

    return service_times

### Confidence interval function

In [32]:
def confidence_interval_service(service_function):
    reps_list = []

    for i in range(n_repetitions):
        t_service = service_function(n_customers)
        blocked_fraction = blocking_with_service_times(
            t_service,
            n_customers,
            m,
            mean_arrival_time=mean_interarrival_time
        )
        reps_list.append(blocked_fraction)

    reps = np.array(reps_list)

    mean = np.mean(reps)
    conf_int = stats.t.interval(
        confidence=0.95,
        df=n_repetitions - 1,
        loc=mean,
        scale=np.std(reps, ddof=1) / np.sqrt(n_repetitions)
    )

    return mean, conf_int

### Blocking function, with varying service time

In [31]:
def blocking_with_service_times(t_service, customers, units, mean_arrival_time=1):
    t_interArrival = np.random.exponential(mean_arrival_time, customers)
    t_arrival = np.cumsum(t_interArrival)

    service_units = [0] * units
    n_blocked = 0

    for i in range(customers):
        arrival_time = t_arrival[i]
        departure_time = arrival_time + t_service[i]

        vacancies = [x for x in service_units if x < arrival_time]

        if len(vacancies) > 0:
            indx = service_units.index(vacancies[0])
            service_units[indx] = departure_time
        else:
            n_blocked += 1

    return n_blocked / customers

### Comparisons

In [33]:
mean_const, ci_const = confidence_interval_service(constant_service)
mean_pareto_105, ci_pareto_105 = confidence_interval_service(lambda n: pareto_service(n, k=1.05))
mean_pareto_205, ci_pareto_205 = confidence_interval_service(lambda n: pareto_service(n, k=2.05))
mean_erlang, ci_erlang = confidence_interval_service(erlang_service)
mean_hyper, ci_hyper = confidence_interval_service(hyperexponential_service)

print("Constant service")
print("Mean blocking:", mean_const)
print("CI:", ci_const)

print("\nPareto service, k = 1.05")
print("Mean blocking:", mean_pareto_105)
print("CI:", ci_pareto_105)

print("\nPareto service, k = 2.05")
print("Mean blocking:", mean_pareto_205)
print("CI:", ci_pareto_205)

print("\nErlang service")
print("Mean blocking:", mean_erlang)
print("CI:", ci_erlang)

print("\nHyperexponential service")
print("Mean blocking:", mean_hyper)
print("CI:", ci_hyper)


Constant service
Mean blocking: 0.12161
CI: (np.float64(0.11779484513692152), np.float64(0.12542515486307845))

Pareto service, k = 1.05
Mean blocking: 0.0016800000000000003
CI: (np.float64(0.00029254360678281945), np.float64(0.003067456393217181))

Pareto service, k = 2.05
Mean blocking: 0.12126999999999999
CI: (np.float64(0.11460432617020479), np.float64(0.12793567382979518))

Erlang service
Mean blocking: 0.12147000000000001
CI: (np.float64(0.11860022877446677), np.float64(0.12433977122553325))

Hyperexponential service
Mean blocking: 0.11817
CI: (np.float64(0.11450868156852592), np.float64(0.12183131843147407))


## Part 4

### Setup

In [34]:
mean_arrival_erlang, ci_arrival_erlang = confidence_interval(erlang_interarrival)
mean_arrival_hyper, ci_arrival_hyper = confidence_interval(hyperexponential_interarrival)

mean_service_const, ci_service_const = confidence_interval_service(constant_service)
mean_service_pareto_105, ci_service_pareto_105 = confidence_interval_service(lambda n: pareto_service(n, k=1.05))
mean_service_pareto_205, ci_service_pareto_205 = confidence_interval_service(lambda n: pareto_service(n, k=2.05))
mean_service_erlang, ci_service_erlang = confidence_interval_service(erlang_service)
mean_service_hyper, ci_service_hyper = confidence_interval_service(hyperexponential_service)

### Comparison

In [35]:
comparison = [
    ["Poisson arrivals, exponential service", mean, conf_int],
    ["Erlang interarrival, exponential service", mean_arrival_erlang, ci_arrival_erlang],
    ["Hyperexponential interarrival, exponential service", mean_arrival_hyper, ci_arrival_hyper],
    ["Poisson arrivals, constant service", mean_service_const, ci_service_const],
    ["Poisson arrivals, Pareto service k=1.05", mean_service_pareto_105, ci_service_pareto_105],
    ["Poisson arrivals, Pareto service k=2.05", mean_service_pareto_205, ci_service_pareto_205],
    ["Poisson arrivals, Erlang service", mean_service_erlang, ci_service_erlang],
    ["Poisson arrivals, hyperexponential service", mean_service_hyper, ci_service_hyper],
]

for name, estimate, ci in comparison:
    print(name)
    print("Mean blocked fraction:", estimate)
    print("95% CI:", ci)
    print()

Poisson arrivals, exponential service
Mean blocked fraction: 0.12127999999999999
95% CI: [np.float64(0.11599713243563459), np.float64(0.12656286756436538)]

Erlang interarrival, exponential service
Mean blocked fraction: 0.06753999999999999
95% CI: (np.float64(0.06494860376888267), np.float64(0.0701313962311173))

Hyperexponential interarrival, exponential service
Mean blocked fraction: 0.13873
95% CI: (np.float64(0.13471208932498344), np.float64(0.14274791067501655))

Poisson arrivals, constant service
Mean blocked fraction: 0.12267000000000002
95% CI: (np.float64(0.11761008372926465), np.float64(0.12772991627073538))

Poisson arrivals, Pareto service k=1.05
Mean blocked fraction: 0.00131
95% CI: (np.float64(0.00023170508570621276), np.float64(0.0023882949142937874))

Poisson arrivals, Pareto service k=2.05
Mean blocked fraction: 0.11882999999999999
95% CI: (np.float64(0.11185804858742861), np.float64(0.12580195141257136))

Poisson arrivals, Erlang service
Mean blocked fraction: 0.120